In [1]:
from pyspark.sql import SparkSession

KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"

spark = (
    SparkSession.builder
    .appName("PD4-Kafka-Streaming")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 4.0.0-preview2


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, col, to_timestamp

kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

tx_schema = StructType([
    StructField("tx_id", StringType()),
    StructField("user_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("store", StringType()),
    StructField("category", StringType()),
    StructField("timestamp", StringType())
])

df = (
    kafka_raw
    .select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*")
    .withColumn("timestamp", to_timestamp("timestamp"))
)

df.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- store: string (nullable = true)
 |-- category: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [3]:
from pyspark.sql.functions import (
    window, count, sum as _sum, round as _round,
    current_timestamp, unix_timestamp
)

wyniki_pd = (
    df
    .withWatermark("timestamp", "10 seconds")
    .groupBy(window("timestamp", "2 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "store",
        "liczba_tx",
        "suma_PLN",
        current_timestamp().alias("processing_time"),
        (
            unix_timestamp(current_timestamp()) - unix_timestamp(col("window.end"))
        ).alias("opoznienie_sekundy")
    )
)

In [4]:
query_pd = (
    wyniki_pd
    .writeStream
    .format("memory")
    .queryName("wyniki_pd")
    .outputMode("complete")
    .start()
)

print("Stream pracy domowej uruchomiony:", query_pd.isActive)

Stream pracy domowej uruchomiony: True


In [8]:
spark.sql("""
SELECT *
FROM wyniki_pd
ORDER BY window_start, store
""").show(truncate=False)

+-------------------+-------------------+--------+---------+--------+-----------------------+------------------+
|window_start       |window_end         |store   |liczba_tx|suma_PLN|processing_time        |opoznienie_sekundy|
+-------------------+-------------------+--------+---------+--------+-----------------------+------------------+
|2026-05-01 09:24:00|2026-05-01 09:26:00|Gdańsk  |11       |24533.26|2026-05-01 09:24:59.434|-61               |
|2026-05-01 09:24:00|2026-05-01 09:26:00|Kraków  |13       |30602.98|2026-05-01 09:24:59.434|-61               |
|2026-05-01 09:24:00|2026-05-01 09:26:00|Warszawa|19       |48047.66|2026-05-01 09:24:59.434|-61               |
|2026-05-01 09:24:00|2026-05-01 09:26:00|Wrocław |15       |29969.72|2026-05-01 09:24:59.434|-61               |
+-------------------+-------------------+--------+---------+--------+-----------------------+------------------+



In [10]:
query_pd.stop()